<a href="https://colab.research.google.com/github/anasmita3/SURGE/blob/main/SURGE_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from huggingface_hub import list_repo_files

files = list_repo_files(
    "ai4bharat/IndicCorpV2",
    repo_type="dataset"
)

print(len(files))
print(files[:20])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


28
['.gitattributes', 'README.md', 'data/as.txt', 'data/bd.txt', 'data/bn.txt', 'data/dg.txt', 'data/en.txt', 'data/gom.txt', 'data/gu.txt', 'data/hi-1.txt', 'data/hi-2.txt', 'data/hi-3.txt', 'data/kha.txt', 'data/kn.txt', 'data/ks.txt', 'data/mai.txt', 'data/ml.txt', 'data/mni.txt', 'data/mr.txt', 'data/ne.txt']


In [2]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)

README.md:   0%|          | 0.00/4.01k [00:00<?, ?B/s]

data/bn.txt:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Loading dataset shards:   0%|          | 0/33 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 41004792
    })
})


In [3]:
from datasets import load_dataset

dataset = load_dataset(
    "ai4bharat/IndicCorpV2",
    data_files={"train": "data/bn.txt"}
)

print(dataset)
print(dataset["train"][0])

Loading dataset shards:   0%|          | 0/33 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 41004792
    })
})
{'text': 'নয়াদিল্লি: আজ ভ্যালেন্টাইনস ডে। এমন একটি দিনে প্রিয় মানুষকে হৃদয় উজাড় করা ভালোবাসার বার্তা দিতে পৌঁছে দিতে উন্মুখ অনেকেই। এমন একটি দিনে ভারতীয় দলের প্রাক্তন ক্রিকেটার সচিন তেন্ডুলকর জানালেন তাঁর প্রথম ভালোবাসার কথা। একটি ভিডিও পোস্ট করেছেন ভারতীয় দলের প্রাক্তন ব্যাটিং স্তম্ভ। ভিডিওতে আন্তর্জাতিক ক্রিকেট ১০০ শতরানের মালিককে নেটে অনুশীলন করতে দেখা যাচ্ছে। ওই ভিডিও শেয়ার করে সচিন লিখেছেন, আমার প্রথম ভালোবাসা।'}


In [ ]:
count = 0

for text in dataset["train"]["text"]:
    if text and text.strip():
        count += 1

print("Non-empty docs:", count)

Non-empty docs: 20502396


In [4]:
import kagglehub

path = kagglehub.dataset_download(
    "mdsalmanhossain/bangla-social-media-abuse-dataset"
)

print(path)

100%|██████████| 4.44M/4.44M [00:00<00:00, 6.38MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/mdsalmanhossain/bangla-social-media-abuse-dataset/versions/1


In [5]:
import pandas as pd
import os

file = os.path.join(path, os.listdir(path)[0])

df = pd.read_excel(file)

print(df.columns.tolist())
print("Rows:", len(df))

['comment', 'Category', 'Gender', 'comment react number', 'label']
Rows: 44001


In [6]:
import random

TARGET = 62617
random.seed(42)

reservoir = []
seen = 0

for text in dataset["train"]["text"]:

    if not text or not text.strip():
        continue

    seen += 1

    if len(reservoir) < TARGET:
        reservoir.append(text.strip())
    else:
        j = random.randint(0, seen - 1)

        if j < TARGET:
            reservoir[j] = text.strip()

print("News:", len(reservoir))

News: 62617


In [7]:
social_docs = (
    df["comment"]
    .dropna()
    .astype(str)
    .tolist()
)

social_62617 = []

while len(social_62617) < 62617:
    social_62617.extend(social_docs)

social_62617 = social_62617[:62617]

print("Social:", len(social_62617))

Social: 62617


In [8]:
combined = reservoir + social_62617

random.shuffle(combined)

print("Total:", len(combined))

Total: 125234


In [9]:
with open(
    "news_social_125k.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in combined:
        f.write(doc.strip() + "\n")

print("Saved")

Saved


In [10]:
import os

print(
    "Documents:",
    sum(
        1
        for _ in open(
            "news_social_125k.txt",
            encoding="utf-8"
        )
    )
)

print(
    "Size MB:",
    round(
        os.path.getsize(
            "news_social_125k.txt"
        )/(1024**2),
        2
    )
)

Documents: 125402
Size MB: 61.88


In [11]:
from collections import Counter

chars = 0
words = 0
vocab = Counter()

with open(
    "news_social_125k.txt",
    encoding="utf-8"
) as f:

    for line in f:

        chars += len(line)

        toks = line.split()

        words += len(toks)

        vocab.update(toks)

print("Characters :", chars)
print("Words      :", words)
print("Vocabulary :", len(vocab))
print("TTR        :", len(vocab)/words)
print(
    "Average Word Length:",
    chars/words
)

Characters : 24600273
Words      : 3893200
Vocabulary : 343964
TTR        : 0.08834994349121546
Average Word Length: 6.318779667111888


In [12]:
from sklearn.model_selection import train_test_split

with open(
    "news_social_125k.txt",
    encoding="utf-8"
) as f:

    docs = [
        x.strip()
        for x in f
        if x.strip()
    ]

train_docs, test_docs = train_test_split(
    docs,
    test_size=0.10,
    random_state=42
)

print("Train:", len(train_docs))
print("Test :", len(test_docs))

Train: 112860
Test : 12541


In [13]:
with open(
    "train_news_social_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in train_docs:
        f.write(doc + "\n")

with open(
    "test_news_social_bn.txt",
    "w",
    encoding="utf-8"
) as f:

    for doc in test_docs:
        f.write(doc + "\n")

print("Saved")

Saved


In [14]:
import os

print(
    "Train MB:",
    round(
        os.path.getsize(
            "train_news_social_bn.txt"
        )/(1024**2),
        2
    )
)

print(
    "Test MB:",
    round(
        os.path.getsize(
            "test_news_social_bn.txt"
        )/(1024**2),
        2
    )
)

Train MB: 55.63
Test MB: 6.25


Transliteration

In [17]:
!pip install aksharamukha
from aksharamukha import transliterate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 74.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 48.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.9/531.9 kB 37.6 MB/s eta 0:00:00


In [18]:
def transliterate_file(
    infile,
    outfile
):

    count = 0

    with open(
        infile,
        encoding="utf-8"
    ) as fin, open(
        outfile,
        "w",
        encoding="utf-8"
    ) as fout:

        for line in fin:

            line = line.strip()

            if not line:
                continue

            iso = transliterate.process(
                "Bengali",
                "ISO",
                line
            )

            fout.write(
                iso + "\n"
            )

            count += 1

            if count % 10000 == 0:

                print(
                    f"{count:,} docs"
                )

    print(
        "Completed:",
        count
    )

In [19]:
transliterate_file(
    "train_news_social_bn.txt",
    "train_news_social_iso.txt"
)

10,000 docs
20,000 docs
30,000 docs
40,000 docs
50,000 docs
60,000 docs
70,000 docs
80,000 docs
90,000 docs
100,000 docs
110,000 docs
Completed: 112860


In [21]:
transliterate_file(
    "test_news_social_bn.txt",
    "test_news_social_iso.txt"
)

10,000 docs
Completed: 12541


In [22]:
import random

with open(
    "test_news_social_bn.txt",
    encoding="utf-8"
) as f:

    bn_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

with open(
    "test_news_social_iso.txt",
    encoding="utf-8"
) as f:

    iso_lines = [
        x.strip()
        for x in f
        if x.strip()
    ]

random.seed(42)

idxs = random.sample(
    range(len(bn_lines)),
    100
)

correct = 0

for idx in idxs:

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if back == bn_lines[idx]:
        correct += 1

print(
    "Exact Match:",
    correct,
    "/ 100"
)

print(
    "Accuracy:",
    correct/100
)

Exact Match: 52 / 100
Accuracy: 0.52


In [23]:
import random

random.seed(42)

shown = 0

for idx in idxs:

    original = bn_lines[idx]

    back = transliterate.process(
        "ISO",
        "Bengali",
        iso_lines[idx]
    )

    if original != back:

        print("="*80)
        print("ORIGINAL:")
        print(original)

        print("\nBACK:")
        print(back)

        shown += 1

        if shown == 10:
            break

ORIGINAL:
আলম ভাই ভালো মানুষ।

BACK:
আলম ভাই ভালো মানুষ।
ORIGINAL:
“আমি শুধু সময়ের দিকে সবার দৃষ্টি আকর্ষণ করতে চাইছি। প্রতি মুহূর্তে যেন আমরা সুযোগ হারিয়ে ফেলছি। এখনো বিষয়টা বুঝে উঠতে পারলে আমাদের সামনে যেরকমের যুদ্ধ আমরা সেরকমের প্রস্তুতি নিতে পারতাম। যুদ্ধটার চেহারাটা যেন আমরা কেউ দেশের মানুষের কাছে তুলে ধরতে পারছি না। চেহারাটা পরিষ্কার বুঝতে পারলে সহজে সিদ্ধান্ত নিতে পারতাম যে— জীবনের উপরেই যখন হামলা, জীবন বাজী রেখেই এখন লড়াইতে নামবো। আত্মসমর্পনের কোনো সুযোগ এখানে নেই।

BACK:
“আমি শুধু সময়ের দিকে সবার দৃষ্টি আকর্ষণ করতে চাইছি। প্রতি মুহূর্তে যেন আমরা সুয়োগ হারিয়ে ফেলছি। এখনো বিষয়টা বুঝে উঠতে পারলে আমাদের সামনে যেরকমের যুদ্ধ আমরা সেরকমের প্রস্তুতি নিতে পারতাম। যুদ্ধটার চেহারাটা যেন আমরা কেউ দেশের মানুষের কাছে তুলে ধরতে পারছি না। চেহারাটা পরিষ্কার বুঝতে পারলে সহজে সিদ্ধান্ত নিতে পারতাম যে— জীবনের উপরেই যখন হামলা, জীবন বাজী রেখেই এখন লড়াইতে নামবো। আত্মসমর্পনের কোনো সুয়োগ এখানে নেই।
ORIGINAL:
এছাড়া, এর মধ্য দিয়ে বাংলাদেশ ও নাইজেরিয়ার মধ্যকার বিদ্যমান সৌহার্দ্যপূর্ণ সম্পর্ক আরও জোরদার 

In [24]:
import os
from collections import Counter

def corpus_stats(path):

    chars = 0
    words = 0
    vocab = Counter()

    with open(path, encoding="utf-8") as f:

        for line in f:

            chars += len(line)

            toks = line.split()

            words += len(toks)

            vocab.update(toks)

    return {
        "chars": chars,
        "words": words,
        "vocab": len(vocab),
        "avg_word_len": chars / words
    }

bn = corpus_stats("train_news_social_bn.txt")
iso = corpus_stats("train_news_social_iso.txt")

print("Bengali:", bn)
print("ISO:", iso)

print(
    "\nExpansion Factor:",
    iso["chars"] / bn["chars"]
)

print(
    "Vocabulary Ratio:",
    iso["vocab"] / bn["vocab"]
)

Bengali: {'chars': 22115321, 'words': 3500357, 'vocab': 321674, 'avg_word_len': 6.318018704949238}
ISO: {'chars': 25781441, 'words': 3500323, 'vocab': 309388, 'avg_word_len': 7.365446274529522}

Expansion Factor: 1.1657728594579297
Vocabulary Ratio: 0.9618060520900041


Tokenization

In [26]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,
    10000,
    15000,
    20000,
    25000,
    30000,
    35000,
    40000,
    45000,
    50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = (
        Whitespace()
    )

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000
Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [27]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [28]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_bn_{vocab_size}.json",
        "test_news_social_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 9.927630121957118e-05, 'fertility': 1.774309329681323, 'cpt': 2.997193787884222, 'compression': 2.997193787884222, 'avg_tokens': np.float64(1.774309329681323), 'wfr': 0.45674989754176604, 'variance': np.float64(1.6974084799242335)}
{'vocab': 10000, 'oov': 9.927630121957118e-05, 'fertility': 1.5384568389916582, 'cpt': 3.456677344417015, 'compression': 3.456677344417015, 'avg_tokens': np.float64(1.5384568389916582), 'wfr': 0.34793034367419046, 'variance': np.float64(1.239532747955121)}
{'vocab': 15000, 'oov': 9.927630121957118e-05, 'fertility': 1.4429937659573926, 'cpt': 3.6853581950006173, 'compression': 3.6853581950006173, 'avg_tokens': np.float64(1.4429937659573926), 'wfr': 0.29857220314476773, 'variance': np.float64(1.043704797824405)}
{'vocab': 20000, 'oov': 9.927630121957118e-05, 'fertility': 1.3910366227729654, 'cpt': 3.8230114244618534, 'compression': 3.8230114244618534, 'avg_tokens': np.float64(1.3910366227729654), 'wfr': 0.27013081561845315, 'variance': n

In [29]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        BPE(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = BpeTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"bpe_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [30]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [31]:
for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"bpe_iso_{vocab_size}.json",
        "test_news_social_iso.txt"
    )

    print(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 9.92773120795033e-05, 'fertility': 1.7471763241429694, 'cpt': 3.6384676240859375, 'compression': 3.6384676240859375, 'avg_tokens': np.float64(1.7471763241429694), 'wfr': 0.44708137430346784, 'variance': np.float64(1.5753649849876301)}
{'vocab': 10000, 'oov': 9.92773120795033e-05, 'fertility': 1.5226594100891204, 'cpt': 4.174961548749499, 'compression': 4.174961548749499, 'avg_tokens': np.float64(1.5226594100891204), 'wfr': 0.3397880556665708, 'variance': np.float64(1.1658212327724378)}
{'vocab': 15000, 'oov': 9.92773120795033e-05, 'fertility': 1.4323959688320151, 'cpt': 4.438049692465448, 'compression': 4.438049692465448, 'avg_tokens': np.float64(1.4323959688320151), 'wfr': 0.2919338456721456, 'variance': np.float64(1.005155689588499)}
{'vocab': 20000, 'oov': 9.92773120795033e-05, 'fertility': 1.383228243631615, 'cpt': 4.595802983514481, 'compression': 4.595802983514481, 'avg_tokens': np.float64(1.383228243631615), 'wfr': 0.2645791278360855, 'variance': np.float6

In [32]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_bn.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_bn_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [33]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [34]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_bn_{vocab_size}.json",
        "test_news_social_bn.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 9.927630121957118e-05, 'fertility': 1.9325175706325428, 'cpt': 2.7518243463966194, 'compression': 2.7518243463966194, 'avg_tokens': np.float64(1.9325175706325428), 'wfr': 0.5109394847305412, 'variance': np.float64(2.16950038768024)}
{'vocab': 10000, 'oov': 9.927630121957118e-05, 'fertility': 1.6129242470910772, 'cpt': 3.2970853468765484, 'compression': 3.2970853468765484, 'avg_tokens': np.float64(1.6129242470910772), 'wfr': 0.3768732037989731, 'variance': np.float64(1.4667061930915637)}
{'vocab': 15000, 'oov': 9.927630121957118e-05, 'fertility': 1.4944112533505751, 'cpt': 3.558557852654458, 'compression': 3.558557852654458, 'avg_tokens': np.float64(1.4944112533505751), 'wfr': 0.31993951782264163, 'variance': np.float64(1.2220517608987083)}
{'vocab': 20000, 'oov': 9.927630121957118e-05, 'fertility': 1.429624557393157, 'cpt': 3.71982201425168, 'compression': 3.71982201425168, 'avg_tokens': np.float64(1.429624557393157), 'wfr': 0.2867481411148983, 'variance': np.flo

In [35]:
from tokenizers import Tokenizer
from tokenizers.models import WordPiece
from tokenizers.trainers import WordPieceTrainer
from tokenizers.pre_tokenizers import Whitespace

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    tokenizer = Tokenizer(
        WordPiece(
            unk_token="[UNK]"
        )
    )

    tokenizer.pre_tokenizer = Whitespace()

    trainer = WordPieceTrainer(
        vocab_size=vocab_size,
        special_tokens=["[UNK]"]
    )

    tokenizer.train(
        ["train_news_social_iso.txt"],
        trainer
    )

    tokenizer.save(
        f"wp_iso_{vocab_size}.json"
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [36]:
import json
import numpy as np
from collections import Counter
from tokenizers import Tokenizer

def evaluate_tokenizer(
    tokenizer_path,
    test_file
):

    tok = Tokenizer.from_file(
        tokenizer_path
    )

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    vocab = tok.get_vocab()

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                enc = tok.encode(w)

                tks = enc.tokens

                words += 1

                tokens += len(tks)

                chars += len(w)

                lengths.append(
                    len(tks)
                )

                if len(tks) > 1:
                    fragmented += 1

                if "[UNK]" in tks:
                    oov += 1

    return {
        "vocab": len(vocab),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [37]:
results = []

for vocab_size in VOCABS:

    r = evaluate_tokenizer(
        f"wp_iso_{vocab_size}.json",
        "test_news_social_iso.txt"
    )

    print(r)

    results.append(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 9.92773120795033e-05, 'fertility': 1.9125239601974346, 'cpt': 3.3239031882807986, 'compression': 3.3239031882807986, 'avg_tokens': np.float64(1.9125239601974346), 'wfr': 0.4993012404572866, 'variance': np.float64(2.055355943193585)}
{'vocab': 10000, 'oov': 9.92773120795033e-05, 'fertility': 1.6018394304027859, 'cpt': 3.9685903395230944, 'compression': 3.9685903395230944, 'avg_tokens': np.float64(1.6018394304027859), 'wfr': 0.36915122989316235, 'variance': np.float64(1.4118494111521287)}
{'vocab': 15000, 'oov': 9.92773120795033e-05, 'fertility': 1.4875050593245578, 'cpt': 4.27362881834517, 'compression': 4.27362881834517, 'avg_tokens': np.float64(1.4875050593245578), 'wfr': 0.31294245225143125, 'variance': np.float64(1.174304024253433)}
{'vocab': 20000, 'oov': 9.92773120795033e-05, 'fertility': 1.4246370650571862, 'cpt': 4.4622203401036, 'compression': 4.4622203401036, 'avg_tokens': np.float64(1.4246370650571862), 'wfr': 0.2808962450265885, 'variance': np.float64(

In [38]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    spm.SentencePieceTrainer.train(
        input="train_news_social_bn.txt",
        model_prefix=f"uni_bn_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [39]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_path,
    test_file
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                pieces = sp.encode(
                    w,
                    out_type=str
                )

                words += 1
                tokens += len(pieces)
                chars += len(w)

                lengths.append(
                    len(pieces)
                )

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

    return {
        "vocab": sp.get_piece_size(),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [40]:
for vocab_size in VOCABS:

    r = evaluate_unigram(
        f"uni_bn_{vocab_size}.model",
        "test_news_social_bn.txt"
    )

    print(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 0.0, 'fertility': 1.8080327255417559, 'cpt': 2.9412901799730524, 'compression': 2.9412901799730524, 'avg_tokens': np.float64(1.8080327255417559), 'wfr': 0.43132498224481536, 'variance': np.float64(2.100503692139813)}
{'vocab': 10000, 'oov': 0.0, 'fertility': 1.56759825171888, 'cpt': 3.3924182390958397, 'compression': 3.3924182390958397, 'avg_tokens': np.float64(1.56759825171888), 'wfr': 0.34194321904679476, 'variance': np.float64(1.4351627612722622)}
{'vocab': 15000, 'oov': 0.0, 'fertility': 1.4757982196449981, 'cpt': 3.6034390270406393, 'compression': 3.6034390270406393, 'avg_tokens': np.float64(1.4757982196449981), 'wfr': 0.3020545103260081, 'variance': np.float64(1.180569977251153)}
{'vocab': 20000, 'oov': 0.0, 'fertility': 1.4255796845050057, 'cpt': 3.7303764626474702, 'compression': 3.7303764626474702, 'avg_tokens': np.float64(1.4255796845050057), 'wfr': 0.2798038911218986, 'variance': np.float64(1.0434881997804364)}
{'vocab': 25000, 'oov': 0.0, 'fertility':

In [41]:
import sentencepiece as spm

VOCABS = [
    5000,10000,15000,20000,25000,
    30000,35000,40000,45000,50000
]

for vocab_size in VOCABS:

    spm.SentencePieceTrainer.train(
        input="train_news_social_iso.txt",
        model_prefix=f"uni_iso_{vocab_size}",
        vocab_size=vocab_size,
        model_type="unigram",
        character_coverage=1.0,
        unk_id=0
    )

    print(
        "Saved:",
        vocab_size
    )

Saved: 5000
Saved: 10000
Saved: 15000
Saved: 20000
Saved: 25000
Saved: 30000
Saved: 35000
Saved: 40000
Saved: 45000
Saved: 50000


In [42]:
import sentencepiece as spm
import numpy as np

def evaluate_unigram(
    model_path,
    test_file
):

    sp = spm.SentencePieceProcessor()
    sp.load(model_path)

    words = 0
    tokens = 0
    chars = 0

    fragmented = 0
    oov = 0

    lengths = []

    with open(
        test_file,
        encoding="utf-8"
    ) as f:

        for line in f:

            ws = line.split()

            for w in ws:

                pieces = sp.encode(
                    w,
                    out_type=str
                )

                words += 1
                tokens += len(pieces)
                chars += len(w)

                lengths.append(
                    len(pieces)
                )

                if len(pieces) > 1:
                    fragmented += 1

                if "<unk>" in pieces:
                    oov += 1

    return {
        "vocab": sp.get_piece_size(),
        "oov": oov / words,
        "fertility": tokens / words,
        "cpt": chars / tokens,
        "compression": chars / tokens,
        "avg_tokens": np.mean(lengths),
        "wfr": fragmented / words,
        "variance": np.var(lengths)
    }

In [43]:
for vocab_size in VOCABS:

    r = evaluate_unigram(
        f"uni_iso_{vocab_size}.model",
        "test_news_social_iso.txt"
    )

    print(r)

print("\nFinished.")

{'vocab': 5000, 'oov': 0.0, 'fertility': 1.8492410376770128, 'cpt': 3.437650560118053, 'compression': 3.437650560118053, 'avg_tokens': np.float64(1.8492410376770128), 'wfr': 0.4326377981819524, 'variance': np.float64(2.1156897640391703)}
{'vocab': 10000, 'oov': 0.0, 'fertility': 1.6030205758593215, 'cpt': 3.965666183389998, 'compression': 3.965666183389998, 'avg_tokens': np.float64(1.6030205758593215), 'wfr': 0.34328821731039943, 'variance': np.float64(1.5153038669395993)}
{'vocab': 15000, 'oov': 0.0, 'fertility': 1.503750900496132, 'cpt': 4.2274584752476585, 'compression': 4.2274584752476585, 'avg_tokens': np.float64(1.503750900496132), 'wfr': 0.30583012379117147, 'variance': np.float64(1.2539952068102171)}
{'vocab': 20000, 'oov': 0.0, 'fertility': 1.450892095744058, 'cpt': 4.381472962469757, 'compression': 4.381472962469757, 'avg_tokens': np.float64(1.450892095744058), 'wfr': 0.2854986393917101, 'variance': np.float64(1.1068717333692595)}
{'vocab': 25000, 'oov': 0.0, 'fertility': 1.4

## News + Social Media Corpus

The News+Social corpus combines formal news articles with informal social media comments, providing a balanced representation of structured and user-generated Bengali text. The final corpus contained 125,234 documents, evenly distributed between the two domains. The corpus consisted of approximately 3.89 million words and 343,964 unique vocabulary items, with a type-token ratio (TTR) of 0.0883. Compared to the standalone social media corpus, the mixed corpus exhibited a lower TTR due to the higher frequency of recurring lexical items inherited from the news domain.

Transliteration from Bengali script to ISO 15919 increased corpus size by approximately 16.6%, producing an expansion factor of 1.166. The ISO representation reduced the observed vocabulary size by roughly 3.8%, resulting in a vocabulary ratio of 0.962. This behavior closely matches observations from the news and literature corpora, indicating that script transformation primarily affects character-level representation rather than lexical diversity.

A transliteration sanity check produced an exact-match accuracy of 52%. Manual inspection revealed that most mismatches were caused by orthographic normalization, punctuation substitutions, Unicode representation differences, and alternate spellings rather than semantic corruption. Therefore, the exact-match score underestimates the practical fidelity of the transliteration process.

Across all vocabulary sizes, BPE consistently achieved the best intrinsic performance among the evaluated tokenization methods. At a vocabulary size of 50,000, BPE produced the lowest fertility (1.283), lowest word fragmentation ratio (0.208), and highest characters-per-token value (4.145) in Bengali script. WordPiece showed slightly higher fertility and fragmentation, while Unigram exhibited the highest fragmentation and lowest compression efficiency.

The ISO representation demonstrated behavior similar to the Bengali script. Fertility and fragmentation metrics remained nearly unchanged across scripts, while CPT values increased substantially due to the longer character sequences introduced by ISO transliteration. This pattern suggests that script representation primarily influences token length and compression characteristics rather than segmentation behavior.

Overall, the News+Social corpus reinforced the trends observed in previous experiments. BPE remained the strongest tokenizer across all intrinsic metrics, WordPiece consistently ranked second, and Unigram produced the least compact segmentations. The results further indicate that the superiority of BPE is robust across both formal and informal Bengali text domains.